# 1D Heat Equation (Explicit) - Analytical to Numerical Bridge

This notebook demonstrates the mathematical derivation, numerical implementation, and performance benchmarking of the 1D Heat Equation using an **Explicit Forward Euler** scheme.

## 1. Physics Background

The 1D Heat Equation describes the distribution of heat in a region over time. For a solid rod, it models the spatial and temporal evolution of temperature $T(x, t)$.

## 2. Governing Equation

The 1D heat diffusivity equation is given by:
$$\frac{\partial T}{\partial t} = \alpha \frac{\partial^2 T}{\partial x^2}$$

Where:
- $T$: Temperature
- $t$: Time
- $x$: Spatial coordinate
- $\alpha$: Thermal diffusivity ($m^2/s$)

## 3. Detailed Analytical Trace (Sympy)

To find the analytical solution, we use the method of **Separation of Variables**. Here is the runnable trace of that process.

In [ ]:
import sympy as sp
from IPython.display import display, Math

sp.init_printing()

# 1. Symbolic Definition
x, t, alpha, L = sp.symbols('x t alpha L', real=True, positive=True)
T = sp.Function('T')(x, t)
heat_eq = sp.Eq(T.diff(t), alpha * T.diff(x, x))
display(Math(f"\\text{{PDE: }} {sp.latex(heat_eq)}"))

# 2. Separation of Variables Ansatz: T(x, t) = X(x) * Phi(t)
X = sp.Function('X')(x)
Phi = sp.Function('Phi')(t)
ansatz = X * Phi
sep_eq = sp.Eq(ansatz.diff(t), alpha * ansatz.diff(x, x))
display(Math(f"\\text{{Separation Ansatz: }} {sp.latex(sep_eq)}"))

# Divide both sides and set to separation constant -k^2
lam = sp.symbols('lambda', real=True, positive=True)
ode_spatial = sp.Eq(X.diff(x, x) / X, -lam**2)
ode_temporal = sp.Eq(Phi.diff(t) / (alpha * Phi), -lam**2)
display(Math(f"\\text{{Spatial ODE: }} {sp.latex(ode_spatial)}"))
display(Math(f"\\text{{Temporal ODE: }} {sp.latex(ode_temporal)}"))

# 3. Solve the ODEs using sp.dsolve
sol_spatial = sp.dsolve(ode_spatial, X)
sol_temporal = sp.dsolve(ode_temporal, Phi)
display(Math(f"X(x) = {sp.latex(sol_spatial.rhs)}"))
display(Math(f"\\Phi(t) = {sp.latex(sol_temporal.rhs)}"))

### Eigenvalues and Boundary Conditions

Assuming Homogeneous Dirichlet BCs ($X(0)=0, X(L)=0$)
- $X(0)=0 \implies C_2 = 0$
- $X(L)=0 \implies \sin(\lambda L) = 0 \implies \lambda_n = \frac{n\pi}{L}$

This gives the final fundamental solution term:

In [ ]:
n = sp.symbols('n', integer=True, positive=True)
lam_n = n * sp.pi / L
term_n = sp.sin(lam_n * x) * sp.exp(-alpha * lam_n**2 * t)
display(Math(f"T_n(x, t) = A_n {sp.latex(term_n)}"))

## 4. Numerical Implementation & Comparison

We now implement the numerical solvers (Numpy/JAX) and compare them with the **analytical trace** evaluated as a Fourier series.

In [ ]:
import numpy as np
import jax.numpy as jnp
from jax import jit
import matplotlib.pyplot as plt
import time

# Parameters
nx = 101
L_val = 1.0
x_vals = np.linspace(0, L_val, nx)
dx = L_val / (nx - 1)
alpha_val = 0.01
dt = 0.4 * dx**2 / alpha_val  # Stable time step for Explicit
steps = 500
t_final = steps * dt

# Initial Condition: Gaussian Pulse centered at 0.5
u0 = np.exp(-100 * (x_vals - 0.5)**2)
u0[0] = u0[-1] = 0.0 # Dirichlet BCs

def numpy_solver(u, alpha, dx, dt, steps):
    u_curr = u.copy()
    r = alpha * dt / dx**2
    for _ in range(steps):
        u_next = u_curr.copy()
        u_next[1:-1] = u_curr[1:-1] + r * (u_curr[2:] - 2*u_curr[1:-1] + u_curr[0:-2])
        u_curr = u_next
    return u_curr

@jit
def jax_kernel(u, r):
    return u + r * (jnp.roll(u, -1) - 2*u + jnp.roll(u, 1))

def jax_solver(u, alpha, dx, dt, steps):
    u_curr = jnp.array(u)
    r = alpha * dt / dx**2
    for _ in range(steps):
        u_curr = jax_kernel(u_curr, r)
        u_curr = u_curr.at[0].set(0.0)
        u_curr = u_curr.at[-1].set(0.0)
    return u_curr

def evaluate_analytical(x_vals, t_v, alpha_v, L_v, u_init, n_terms=50):
    T_final = np.zeros_like(x_vals)
    dx_sim = L_v / (len(x_vals)-1)
    for n_val in range(1, n_terms + 1):
        kn = n_val * np.pi / L_v
        # Calculate Fourier Coefficient An via integral of initial condition
        An = (2.0/L_v) * np.sum(u_init * np.sin(kn * x_vals)) * dx_sim
        T_final += An * np.sin(kn * x_vals) * np.exp(-alpha_v * kn**2 * t_v)
    return T_final

u_np = numpy_solver(u0, alpha_val, dx, dt, steps)
u_jax = jax_solver(u0, alpha_val, dx, dt, steps).block_until_ready()
u_analytical = evaluate_analytical(x_vals, t_final, alpha_val, L_val, u0)

# Visualization
plt.figure(figsize=(10, 6))
plt.plot(x_vals, u0, 'k:', label='Initial condition')
plt.plot(x_vals, u_np, 'b-', label='Numerical (Numpy)')
plt.plot(x_vals, u_jax, 'r--', label='Numerical (JAX)')
plt.plot(x_vals, u_analytical, 'go', markersize=4, alpha=0.5, label='Analytical Trace')
plt.title(f"1D Heat Equation Verification (Total Time: {t_final:.2f}s)")
plt.legend()
plt.grid(True)
plt.show()

## 5. C++ Backend (axcnt_cpp)

Finally, we verify that our high-performance C++ engine produces the same results.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../../../bindings/python'))
try:
    import axcnt_cpp
    print("Successfully imported axcnt_cpp")
except ImportError:
    print("axcnt_cpp not found. Ensure it is built.")